# Metronome Compass Calibration

A Python port of RNGReporter's HGSS **Seed to Time** verification panel, built for
gathering Metronome-compass calibration data.

Give it a target datetime + delay and a search window, and it enumerates every candidate
seed nearby.  For each seed it reports:

- **Roamer relocation** — where Raikou / Entei / Latios(Latias) move to when the save is
  reloaded (given where they are now).
- **Elm phone-call sequence** — the `P`/`E`/`K` calls you can read off in-game.

Two sections identify the seed two ways: **Section A** from roamer routes + Elm calls
(`a_seed`), **Section B** from the Metronome battle (`b_seed`).  All logic lives in
`utils/calibration_tools.py`.

## Keyboard fixup

The `2` and `w` keys on my keyboard are flaky, so every `input()` prompt below accepts
`\T` for `2` and `\V` for `w` (substituted before the value is used).  It's applied
automatically at each interactive step — ipykernel resets `input` once per cell, so the
library re-installs the fixup at every prompt.  To add pairs, edit `INPUT_SUBS` in
`utils/calibration_tools.py`.


In [ ]:
%load_ext autoreload
%autoreload 2
import datetime as dt
from utils.calibration_tools import (
    # Section A -- roamer routes + Elm calls
    generate_roamer_candidates_near,
    print_roamer_candidates,
    identify_seed,
    # Section B -- Metronome-compass battle
    generate_candidates_near,
    print_candidates,
    narrow_candidates,
    prompt_magikarp,
    # Persist a run
    save_compass_run,
    # Timer calibration math
    calibrate_timer,
    # Review + apply a model update (deliberate; not automatic on save)
    update_calibration_model,
)

## Section A — Roamer + Elm identification  (→ `a_seed`)

**Configure** your target datetime/delay, the search window, and each roamer's **current**
route (where it is *now*, before the reset) plus whether it's still roaming.  Use `0` for a
current route you don't know or care about.  Variables are `a_`-prefixed so they won't clash
with Section B.

**Identify** (after loading the save, read the roamer map and Elm phone):

1. **Roamer routes** — one number per *roaming* legendary in **R E L** order (e.g. `38 42 11`);
   `.` leaves a roamer unconstrained.  Only roamers marked `present` are expected.
2. If more than one candidate matches, **Elm calls** — type `P`/`E`/`K` as you hear each
   call (matched as a substring, since RNG may advance first); other characters are ignored.
   Type `M` to pick a candidate by number instead.
3. The single surviving row is saved to **`a_seed`** (integer seed is `a_seed["seed"]`).


In [17]:
# --- Section A: roamer / Elm target + current roamer state ---
a_target_time    = dt.datetime(2025, 7, 24, 14, 45, 55)
a_target_delay   = 681
a_seconds_window = 1        # +/- X seconds
a_delay_window   = 60       # +/- Y delays
a_match_parity   = True     # only delays with target_delay's even/odd parity
a_display_limit  = 40       # rows to print (None = all)

# Each roamer's CURRENT route (before the reset) and whether it30 is still roaming.
a_prev_routes = {"r": 43, "e": 45, "l": 6}
a_present     = {"r": True, "e": True, "l": True}

a_candidates = generate_roamer_candidates_near(
    a_target_time, a_target_delay, a_seconds_window, a_delay_window,
    prev_routes=a_prev_routes, present=a_present, match_parity=a_match_parity,
)

# Interactively pin down the seed: roamer routes -> Elm calls -> (M) manual pick.
a_seed = identify_seed(a_candidates, a_present, display_limit=a_display_limit)
# GOALS: EKP, KPEK, PEKK, EKKP

Observed roamer routes (R E L, space-separated, . = any):  29 38 19



Observed R=29 E=38 L=19  ->  3 / 183 candidate(s) match

3 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0B0E02C8  2025-07-24 14:45:54     687    +6   -1   29  38  19   3  PEEPKKPKKPEEPPP
  0x0C0E02C8  2025-07-24 14:45:55     687    +6   +0   29  38  19   3  KKPKKPKPPEKPKEE
  0x0D0E02C8  2025-07-24 14:45:56     687    +6   +1   29  38  19   3  PPKEKEPEEKPPEPE


Elm calls (type P/E/K as heard; M = pick manually):  pke


Elm calls so far: PKE
2 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0C0E02C8  2025-07-24 14:45:55     687    +6   +0   29  38  19   3  KKPKKPKPPEKPKEE
  0x0D0E02C8  2025-07-24 14:45:56     687    +6   +1   29  38  19   3  PPKEKEPEEKPPEPE


Elm calls (type P/E/K as heard; M = pick manually):  k


Elm calls so far: PKEK
1 candidate seed(s)

        Seed                 Time   Delay    dD   ds    R   E   L   #  Elm
  0x0D0E02C8  2025-07-24 14:45:56     687    +6   +1   29  38  19   3  PPKEKEPEEKPPEPE

=== Seed identified: 0x0D0E02C8  2025-07-24 14:45:56  delay=687  R/E/L=29/38/19  Elm=PPKEKEPEEKPPEPE ===


## Section B — Expedition-style Seed Identification  (→ `b_seed`)

Ported from the Metronome Compass Testing notebook.  Generate every candidate seed near a
target `(time, delay)`, each with its precomputed Metronome battle path, then walk the real
battle turn by turn — candidates whose path diverges from what you observe drop out until a
single seed remains, saved as **`b_seed`** (the whole row).

Config uses `b_`-prefixed names so it won't clash with Section A.  The Metronome user's
movepool besides Metronome is `DEFAULT_EXTRA_MOVES` in `utils/calibration_tools.py`; edit it
there if it changes.  (Seeds here use the same year-correct `seed_for` as Section A.)

Magikarp's **level and gender are asked at run time** (they change each battle); the Metronome user's own gender is the stable `b_metronome_user_is_female` config.


In [18]:
# --- Section B: Metronome-compass target ---
# b_target_time     = dt.datetime(2025, 7, 24, 14, 49, 0)
b_timer_delay = 327919
b_target_time = a_target_time + dt.timedelta(milliseconds=b_timer_delay+5000)
b_target_delay    = 20045
b_seconds_window  = 2         # +/- X seconds
b_delay_window    = 2000       # +/- Y delays
b_metronome_only  = False     # True = Metronome-only user; False = + DEFAULT_EXTRA_MOVES
b_metronome_user_is_female = True   # the Metronome user's gender (rarely changes)

# Magikarp's level + gender change every run, so prompt for them at execution time.
# opposite_gender is derived relative to the Metronome user's gender above.
b_magikarp_level, b_opposite_gender = prompt_magikarp(b_metronome_user_is_female)

b_candidates = generate_candidates_near(
    b_target_time, b_target_delay, b_seconds_window, b_delay_window,
    magikarp_level=b_magikarp_level, opposite_gender=b_opposite_gender,
    metronome_only=b_metronome_only,
)

# Walk the real battle turn by turn; candidates diverging from what you observe drop out.
b_seed = narrow_candidates(b_candidates, b_magikarp_level, b_opposite_gender,
                           metronome_only=b_metronome_only)


Magikarp level:  11
Magikarp gender (M/F):  F



20005 / 20005 seeds remain -- next is turn 1
        Seed   Delay    dD  predicted turn 1
  0xF60E4E66   20045    +0  KspM420h           (Ice Shard)
  0xF50E4E66   20045    +0  KspM125h           (Bone Club)
  0xF70E4E66   20045    +0  KspM092h           (Toxic)
  0xF40E4E66   20045    +0  KspM297h           (Feather Dance)
  0xF80E4E66   20045    +0  KspM387            (Last Resort)
  0xF60E4E65   20044    -1  KspM268            (Charge)
  0xF60E4E67   20046    +1  KspM417            (Nasty Plot)
  0xF50E4E65   20044    -1  KspM440h           (Cross Poison)
  0xF50E4E67   20046    +1  KspM122h           (Lick)
  0xF70E4E65   20044    -1  KspM096            (Meditate)
  0xF70E4E67   20046    +1  KspM245h           (Extreme Speed)
  0xF40E4E65   20044    -1  KspM145h           (Bubble)
  0xF40E4E67   20046    +1  KspM450h           (Bug Bite)
  0xF80E4E65   20044    -1  KspM235            (Synthesis)
  0xF80E4E67   20046    +1  KspM073h           (Leech Seed)
  ... and 19990 more

--- 

  Metronome selected? (move name or M###):  Hydro Pump
  Hit, crit, or miss? (h/!/-):  h



38 / 20005 seeds remain -- next is turn 2
        Seed   Delay    dD  predicted turn 2
  0xF80E4E6C   20051    +6  KspM286            (Imprison)
  0xF60E4E76   20061   +16  KspM311h           (Weather Ball)
  0xF80E4ECB   20146  +101  KspM016h           (Gust)
  0xF60E4ED5   20156  +111  KspM004hh          (Comet Punch)
  0xF40E4EDF   20166  +121  KspM458h           (Double Hit)
  0xF60E4D89   19824  -221  KspM359h           (Hammer Arm)
  0xF80E4D7F   19814  -231  KspM372h           (Assurance)
  0xF80E4D20   19719  -326  KspM175h           (Flail)
  0xF80E4FB8   20383  +338  KspM398!           (Poison Jab)
  0xF60E4FC2   20393  +348  KspM386h           (Punishment)
  0xF60E5021   20488  +443  KspM116            (Focus Energy)
  0xF40E502B   20498  +453  KspM103h           (Screech)
  0xF70E507B   20578  +533  KspM007h           (Fire Punch)
  0xF40E4C47   19502  -543  KspM234            (Morning Sun)
  0xF80E4C33   19482  -563  KspM260h           (Flatter)
  ... and 23 more

--- Tur

  Metronome selected? (move name or M###):  Assurance
  Hit, crit, or miss? (h/!/-):  h



1 / 20005 seeds remain -- next is turn 3
        Seed   Delay    dD  predicted turn 3
  0xF80E4D7F   19814  -231  KspM259h           (Torment)

Seed identified: 0xF80E4D7F  time=2025-07-24 14:51:29  delay=19814  dD=-231
Full path: KspM056h KspM372h KspM259h KsphM059h~ FRZM317- FRZM078h FRZM082h FRZM023h FRZM009h FRZM459!
Remaining Metronome moves (turn 3+):
  Turn 3: Torment (M259)
  Turn 4: Blizzard (M059)
  Turn 5: Rock Tomb (M317)
  Turn 6: Stun Spore (M078)
  Turn 7: Dragon Rage (M082)
  Turn 8: Stomp (M023)
  Turn 9: Thunder Punch (M009)
  Turn 10: Roar Of Time (M459)


## Section C — Save the run  (→ `data/compass_runs.jsonl`)

Records this calibration run — both identified seeds (`a_seed`, `b_seed`) plus the metadata
below — as one JSON line appended to `data/compass_runs.jsonl`.

You're prompted for a **run tag** (e.g. `300s Samwise`, `11000d Work`), the **target timer
delay**, the **target timer calibration**, and free-form **notes**.  Leaving the tag / delay
/ calibration blank re-uses the previous run's value (notes never default).  The record is
pretty-printed and confirmed (`y`/`n`) before it's written.

> **Saving no longer touches the shared calibration model.** The chart/expedition read
> `data/calibration_model.json`, and changing it invalidates a precomputed chart (~1 h to
> rebuild) — you often save test runs *while* charting a target from the current model. Apply
> model changes deliberately in **Section E**.

In [19]:
# --- Section C: append this run to data/compass_runs.jsonl ---
# Saving does NOT change the shared calibration model (that would invalidate a precomputed
# chart).  Apply model changes deliberately in Section E below.
run_record = save_compass_run(a_seed, b_seed)

Run tag [Compass Target 3]:  
Target timer delay [327919]:  
Target timer calibration [0]:  
Notes:  



{
  "saved_at": "2026-09-09T13:01:35",
  "tag": "Compass Target 3",
  "target_timer_delay": 327919,
  "target_timer_calibration": 0,
  "fresh_boot": true,
  "prior_battles": 0,
  "notes": "",
  "a_seed": {
    "seed": 219022024,
    "seed_hex": "0x0D0E02C8",
    "time": "2025-07-24T14:45:56",
    "delay": 687,
    "sec_delta": 1,
    "delay_delta": 6,
    "r_route": 29,
    "e_route": 38,
    "l_route": 19,
    "rng_calls": 3,
    "elm": "PPKEKEPEEKPPEPE"
  },
  "b_seed": {
    "seed": 4161686911,
    "seed_hex": "0xF80E4D7F",
    "time": "2025-07-24T14:51:29",
    "delay": 19814,
    "sec_delta": 2,
    "delay_delta": -231,
    "path_str": "KspM056h KspM372h KspM259h KsphM059h~ FRZM317- FRZM078h FRZM082h FRZM023h FRZM009h FRZM459!"
  }
}



Save this run? (y/n):  y


Saved to data/compass_runs.jsonl
Calibration model NOT updated (run update_calibration_model() to review + apply changes).


## Section D — Timer calibration math  (delay/calibration ↔ seed_b frame)

Fits the collected runs to answer: **given a timer countdown, what `F_b` frame will I hit?**
`M = target_timer_delay + target_timer_calibration` (ms, calibration signed).

**Two frame "rates" that are easy to confuse:**

- **within-run *average* rate** `= (F_b − F_a)/(T_b − T_a)` — the mean frames/sec over a run.
  This genuinely **rises with M** (≈56.8 → 58.3 Hz here): every run starts at frame ~670 in
  the slow post-boot region, and the longer the run, the more that slow start dilutes out,
  pulling the average up toward the ~60 Hz ceiling.
- **`dF_b/dM`** — the slope you actually need to turn a commanded `M` into `F_b`. This is the
  *instantaneous* rate at battle time, which by 3–7 min is already near the ceiling (~59.6 Hz)
  and barely changing. So **`F_b` vs `M` is essentially a straight line** — just with slope
  ~59.6, **not** the average rate.

**The models (previous vs new), all shown in the report for comparison:**

- **`within_rate` (previous)** — a line whose slope is the within-run *average* rate. Using
  the average rate as the `F_b`-vs-`M` slope is the bug that made residuals explode (~160
  frames) as `M` left the centroid, tanking predictions far from the calibrated delays.
- **`linear_m` (NEW, recommended ★)** — fits `F_b` directly against `M` (robust Theil–Sen).
  Correct slope → residuals drop to ~30 frames, and predictions hold across the whole range.
- **`quad_m` (NEW, experimental)** — adds an `M²` term to test the rising-rate curvature
  directly. So far it barely changes the RMS and its curvature is within noise (near the
  ceiling there's little left to bend), so it isn't used by default — worth re-checking as
  the sweep extends toward 10 min.

`model["recommended"]` (= `linear_m`) drives the top-level `predict` / `solve` /
`hit_probability`; each individual model is under `model["models"][name]`.

**Uncertainty is still split two ways:** reducible **mean-uncertainty** (shrinks with more
runs; ~0 near measured delays, grows as you extrapolate) and irreducible **physical jitter**
(`σ ≈ c·√M`) — the latter sets your real hit odds, so use a `tolerance` window that reflects
which frames are actually acceptable.


---

**Option 1 vs option 2 — how to *place* `F_b` (year handling):**

- **Option 1 (current, `linear_m` ★)** — fits `F_b` directly; the intercept `α` swallows
  `F_a` *and* the year term as one constant. Correct only for the year you calibrated in —
  a `F_b` prediction used as a seed frame in year 2025 lands `year−2000 = 25` frames early.
- **Option 2 (`linear_df`)** — fits `dF = F_b − F_a`, then reconstructs
  `F_b = dF(M) + F_a(actual)`, where `F_a(actual)` is the low-16 of your *identified* initial
  seed (which already carries the year). **Year-agnostic by construction**, and old runs stay
  reusable across target years.

The report prints an **apples-to-apples verdict** — the `dF` fit's RMS residual *is* option 2's
`F_b`-placement error (`dF − f_dF = (F_a+dF) − (f_dF+F_a)`), so the two RMS numbers compare
directly. On the current data the two are within noise of each other (`F_a` only wobbles ~14
frames, so removing it barely tightens a ~74-frame residual) — i.e. **option 2 costs no
accuracy**; the choice is about the year handling, not the fit quality.


In [20]:
# --- Section D: fit the collected runs; predict the F_b frame from a timer delay ---
model = calibrate_timer()   # reads data/compass_runs.jsonl, prints the model comparison

# === Forward prediction: given a timer delay + calibration, what frame will I land on? ===
d_delay       = 327919    # target_timer_delay (ms)
d_calibration = -0     # target_timer_calibration (ms, signed)

def predict_report(sub, label):
    p = sub["predict"](d_delay, d_calibration)   # range = expected +/- (band + 2*jitter)
    print(f"    [{label:<26}] F_b = {p['expected']:8.1f}   ~95% range "
          f"[{p['lo']:.0f}, {p['hi']:.0f}]   (jitter +/-{p['jitter']:.1f})")

print(f"\nWith delay={d_delay} cal={d_calibration}  (M = {d_delay + d_calibration} ms) "
      f"-> predicted F_b:")
predict_report(model["models"]["within_rate"], "previous within-rate")
predict_report(model["models"]["linear_m"],    "NEW F_b-vs-M line  (used)")
if "quad_m" in model["models"]:
    predict_report(model["models"]["quad_m"],  "NEW quadratic (experimental)")

# === How likely is that delay to land on a specific frame (within a +/- window)? ===
d_target_fb = round(model["predict"](d_delay, d_calibration)["expected"])  # default: the predicted frame
d_tolerance = 25          # half-window (frames) counted as a hit; 0.5 = exactly that frame
hp = model["hit_probability"](d_delay, d_calibration, d_target_fb, tolerance=d_tolerance)
print(f"\nP(land on F_b={d_target_fb} +/-{d_tolerance} at that delay) = {hp['p']*100:.1f}%  "
      f"(expected off by {hp['delta']:+.1f} frames)")

# === Option 2 (dF) models: predict dF, then reconstruct F_b = dF + actual F_a ===
# In the real flow F_a is the low16 of your IDENTIFIED initial seed (a_seed & 0xFFFF); it
# carries the year term, which is what makes option-2 reconstruction year-agnostic. Here we
# use a representative F_a so the reconstructed F_b lines up with the F_b models above.
d_Fa = 679   # your initial-seed low16 (a_seed & 0xFFFF); ~679 on this console
for _k, _lbl in [("linear_df", "dF line (option 2)"), ("quad_df", "dF quad (option 2)")]:
    if _k in model["models"]:
        _p = model["models"][_k]["predict"](d_delay, d_calibration)
        print(f"    [{_lbl:<26}] dF = {_p['expected']:8.1f}  -> F_b = dF + F_a({d_Fa}) "
              f"= {_p['expected'] + d_Fa:8.1f}   (jitter +/-{_p['jitter']:.1f})")

# === (optional) reverse lookup: which delay would center you on a target frame? ===
sol = model["solve"](d_target_fb, calibration=d_calibration)
print(f"\n(reverse) to center on F_b={d_target_fb} holding cal={d_calibration}: "
      f"delay = {sol['delay']:.0f} ms")


=== Timer calibration  (37 run(s), 37 timed, 1 excluded as outlier) ===

  F_b as a function of M = delay + calibration (ms)

  within-run avg rate 58.0673 +/- 0.0126 Hz (dF/dt; rises with M as the slow post-boot frames dilute out)

   within-run-rate slope (previous)       slope 58.0673 Hz                   RMS residual  294.4 frames
  *F_b-vs-M line (NEW)                    slope 59.9979 Hz                   RMS residual   74.4 frames
   F_b-vs-M quadratic (NEW, experimental) inst rate 59.566->60.952 Hz over M range RMS residual   60.8 frames

  ( * = recommended; predict/solve/hit_probability use it )

  RTC-second offset 5.31 +/- 0.52 s  (battle RTC second = round(M/1000 + this); +/-1 s is timestamp truncation)

  tag                     M      Fa      Fb      dF    dt     rate
  11000d Smeagol     185000     673   11469   10796   190   56.821
  11000d Smeagol     180000     651   11166   10515   185   56.838
  11000d Smeagol     176913     677   11013   10336   182   56.791
  1100

## Section E — Apply the model update  (→ `data/calibration_model.json`)

Section D only *reports* the fit; it doesn't change anything. This cell re-fits from all runs,
shows how each parameter differs from the **currently deployed** `data/calibration_model.json`,
and writes the new model **only after you confirm**.

Run it when you actually want the chart/expedition to adopt the latest calibration — **not**
every time you save a test run. Because the chart reads this artifact, applying a change means
the next `x.precompute_chart()` will rebuild (~1 h), so do it deliberately between charting
sessions.

In [ ]:
# --- Section E: review the re-fit vs the deployed model, then write it only if confirmed ---
# Prints an old -> new parameter table and asks before overwriting data/calibration_model.json.
# Applying it means the chart is stale until you re-run x.precompute_chart().
new_model = update_calibration_model()